# NSR 전사 서버 — 구글 콜랩판

폰 대신 콜랩의 무료 GPU가 전사합니다. 폰보다 수십 배 빠릅니다.

**쓰는 법 (처음 3분)**

1. 위 메뉴 **런타임 → 런타임 유형 변경**에서 **T4 GPU** 를 고르십시오.
2. **런타임 → 모두 실행**을 누르십시오. 첫 실행은 모델을 받느라 2~5분 걸립니다.
3. 마지막 셀 출력에 나오는 **주소를 복사**해서, NSR 앱의
   **설정 → 전사 모델 → 노트북·서버로 전사**를 켜고 **주소 칸**에 붙여넣으십시오.
   **모델 칸은 비워 두십시오** — 이 서버는 아래에서 고른 모델로 고정됩니다.

그 다음부터는 앱에서 평소처럼 전사를 누르면 콜랩이 대신 일합니다.
30분 조각 기준 대략 1~3분입니다(추정 — 세션마다 다릅니다).

**알고 쓰십시오**

- 기록 음성 **원본이 구글(콜랩) 서버와 Cloudflare 터널을 지나갑니다.**
  내 컴퓨터가 아닙니다. 이 경로가 싫으면 폰 전사나 내 노트북 서버를 쓰십시오.
- 주소 끝에 무작위 비밀 문자열이 붙어 있어서, 주소를 통째로 모르는 남은 못 씁니다.
  그래도 주소를 다른 곳에 붙여넣지 마십시오.
- 콜랩 무료 세션은 탭을 닫거나 오래 놔두면 꺼집니다. **전사하는 동안 이 탭을
  열어 두십시오.** 꺼졌으면 '모두 실행'을 다시 — 주소가 새로 나오니 앱에도 다시 넣습니다.
- 전사할 때만 켜는 개인용입니다. 상시 서버로 두는 것은 콜랩 이용 규칙과 맞지 않습니다.


In [ ]:
# 필요한 것 설치 + 터널 프로그램 받기 (1~2분)
%pip -q install faster-whisper fastapi uvicorn python-multipart
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("설치 끝. 다음 셀로.")


In [ ]:
# 전사 서버 — NSR 앱이 말하는 OpenAI 호환 형식 (/v1/audio/transcriptions, verbose_json)
import os
import tempfile

from fastapi import FastAPI, File, Form, UploadFile


def build_app(model, secret: str) -> FastAPI:
    app = FastAPI()

    @app.get(f"/{secret}/health")
    def health():
        return {"status": "ok"}

    @app.post(f"/{secret}/v1/audio/transcriptions")
    async def transcribe(
        file: UploadFile = File(...),
        language: str = Form("ko"),
        temperature: float = Form(0.0),
        prompt: str = Form(""),
        response_format: str = Form("verbose_json"),
        model_name: str = Form("", alias="model"),  # 앱이 보내지만 이 서버는 모델 고정
    ):
        suffix = os.path.splitext(file.filename or "audio.m4a")[1] or ".m4a"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
            f.write(await file.read())
            path = f.name
        try:
            segment_iter, info = model.transcribe(
                path,
                language=language or "ko",
                temperature=temperature,
                initial_prompt=prompt or None,
                beam_size=5,
                # 잡음·무음 구간에서 같은 문장이 반복되는 환각을 줄인다.
                condition_on_previous_text=False,
            )
            segments = [
                {"id": i, "start": round(s.start, 2), "end": round(s.end, 2), "text": s.text}
                for i, s in enumerate(segment_iter)
            ]
        finally:
            os.unlink(path)
        return {
            "task": "transcribe",
            "language": info.language,
            "duration": round(info.duration, 2),
            "text": "".join(s["text"] for s in segments).strip(),
            "segments": segments,
        }

    return app


In [ ]:
# 모델을 싣고 서버·터널을 띄운다. 마지막에 나오는 주소를 앱에 넣으면 된다.
import re
import secrets
import subprocess
import threading
import time

import ctranslate2
import uvicorn
from faster_whisper import WhisperModel

# 한국어 파인튜닝판 — 앱의 '노트북에 한국어 모델 쓰기'와 같은 모델이다.
# 이 모델이 안 열리면 "Systran/faster-whisper-large-v3" 로 바꿔 다시 실행하십시오.
MODEL_ID = "ghost613/faster-whisper-large-v3-turbo-korean"
PORT = 8000

gpu = ctranslate2.get_cuda_device_count() > 0
if not gpu:
    print("⚠ GPU 가 안 잡혔습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 고른 뒤")
    print("  '모두 실행'을 다시 하십시오. CPU 로도 되지만 몇 배 느립니다.")
print(f"모델 여는 중: {MODEL_ID} (세션 첫 실행에서만 내려받습니다)")
model = WhisperModel(
    MODEL_ID,
    device="cuda" if gpu else "cpu",
    compute_type="float16" if gpu else "int8",
)

secret = secrets.token_urlsafe(12)
app = build_app(model, secret)
threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"),
    daemon=True,
).start()

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:
        break
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
if not url:
    raise RuntimeError("터널 주소를 못 받았습니다. 이 셀만 한 번 더 실행해 보십시오.")

print()
print("=" * 62)
print("NSR 앱에 넣을 주소 — 설정 → 전사 모델 → 노트북·서버로 전사:")
print()
print(f"    {url}/{secret}")
print()
print("모델 칸은 비워 두십시오. 이 탭을 닫으면 서버도 꺼집니다.")
print("=" * 62)
